In [ ]:
import os
import sys
from pathlib import Path

# Get the project root directory (2 levels up from notebooks)
project_root = str(Path().absolute().parent.parent)
print(f"Adding to path: {project_root}")

# Add to Python path
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
from backend.database.mongodb_jobfocus import mongodb, get_jobs_collection, get_searches_collection

# Test connection
try:
    jobs_collection = get_jobs_collection()
    searches_collection = get_searches_collection()
    print("MongoDB collections accessible")
    
    # Test indexes
    mongodb.ensure_indexes()
    print("Indexes created")
except Exception as e:
    print(f"MongoDB connection error: {e}")

In [ ]:
from backend.utils.job_search import process_job_search

total_jobs, current_page, storage_success = process_job_search(
    job_title="Machine Learning Engineer",
    job_location="San Francisco CA",
    max_page_depth=3
)


------------------- Older tests -------------------

In [ ]:
# # Test API fetch
# from backend.utils.job_search import fetch_jobs_from_api

# response = fetch_jobs_from_api(
#     job_title="Python Developer",
#     job_location="San Francisco"
# )

# if response:
#     print("API Response received")
#     print(f"Response keys: {response.keys()}")
#     print(f"Number of jobs found: {len(response.get('jobs', []))}")
# else:
#     print("API fetch failed")

-------

In [ ]:
# from pydantic import ValidationError
# from backend.models.job_search_models import JobSearchResponse

# # Debug the raw response structure
# print("Raw Response Keys:", response.keys())
# print("\nSample of response data:")
# for key, value in response.items():
#     if isinstance(value, (list, dict)):
#         print(f"{key}: {type(value)} with {len(value)} items")
#     else:
#         print(f"{key}: {value}")

# # Debug the parsing error with more detail
# if response:
#     try:
#         parsed = JobSearchResponse(**response)
#     except ValidationError as e:
#         print("Validation Errors:")
#         for error in e.errors():
#             print(f"Field: {error['loc']}")
#             print(f"Error: {error['msg']}")
#             print(f"Type: {error['type']}\n")

In [ ]:
# Test response parsing
from backend.utils.job_search import parse_jobs_response

# Test parsing with updated models
if response:
    parsed = parse_jobs_response(response)
    if parsed:
        print(f"Found {len(parsed.jobs)} jobs")
        first_job = parsed.jobs[0]
        print("\nFirst job details:")
        print(f"Title: {first_job.title}")
        print(f"Company: {first_job.company_name}")
        print(f"Location: {first_job.location}")
    else:
        print("Failed to parse response")

In [ ]:
from pprint import pprint
pprint(parsed.model_dump())

In [ ]:
# Test different search combinations
test_searches = [
    {"job_title": "Python Developer", "job_location": "Remote"},
    {"job_title": "Data Scientist", "job_location": "San Francisco"},
    {"job_title": "Software Engineer", "job_location": "New York"}
]

for search in test_searches:
    response = fetch_jobs_from_api(**search)
    if response:
        parsed = parse_jobs_response(response)
        print(f"\nSearch for {search['job_title']} in {search['job_location']}:")
        print(f"Found {len(parsed.jobs)} jobs")

In [ ]:
# Delete all documents from both collections
from backend.database.mongodb_jobfocus import get_jobs_collection, get_searches_collection

def clear_collections():
    """Clear all documents from jobs and searches collections."""
    try:
        jobs_collection = get_jobs_collection()
        searches_collection = get_searches_collection()
        
        # Get counts before deletion
        jobs_count = jobs_collection.count_documents({})
        searches_count = searches_collection.count_documents({})
        
        # Delete all documents
        jobs_result = jobs_collection.delete_many({})
        searches_result = searches_collection.delete_many({})
        
        print(f"Deleted {jobs_result.deleted_count}/{jobs_count} documents from job_listings")
        print(f"Deleted {searches_result.deleted_count}/{searches_count} documents from job_searches")
        
        return True
    except Exception as e:
        print(f"Error clearing collections: {e}")
        return False

# Execute the cleanup
clear_collections()

In [ ]:
# Verify collections are empty
jobs_collection = get_jobs_collection()
searches_collection = get_searches_collection()

print(f"Jobs remaining: {jobs_collection.count_documents({})}")
print(f"Searches remaining: {searches_collection.count_documents({})}")

In [ ]:
# Test with detailed error logging
from backend.utils.job_search import fetch_jobs_from_api, parse_jobs_response, store_jobs_in_db

response_page1 = fetch_jobs_from_api(
    job_title="AI Engineer",
    job_location="Palo Alto CA"
)

In [ ]:
response_page1